In [1]:
import bw2analyzer as ba
import bw2data as bd
import bw2calc as bc
import bw2io as bi
import matrix_utils as mu
import bw_processing as bp
import time
import wurst
import os
from pathlib import Path
import copy
import pandas as pd

In [2]:
bd.projects.set_current('Acetylene production')

In [3]:
bd.databases

Databases dictionary with 10 object(s):
	acetylene-production-Base-2020
	acetylene-production-Base-2050
	acetylene-production-RCP19-2050
	acetylene-production-RCP26-2050
	ecoinvent-3.10-biosphere
	ecoinvent-3.10-cutoff
	ecoinvent-3.10-cutoff-Base-2020
	ecoinvent-3.10-cutoff-Base-2050
	ecoinvent-3.10-cutoff-RCP19-2050
	ecoinvent-3.10-cutoff-RCP26-2050

In [4]:
db = bd.Database('acetylene-production-Base-2020')

In [5]:
method = [("IPCC 2021", "climate change", "GWP 100a, incl. H and bio CO2")]

In [38]:
def breakdown_calculations(db, activity):
    activities_list = [{activity : 1}]
    for exchange in activity.technosphere():
        activities_list.append({bd.Database(exchange.input.key[0]).get(exchange.input.key[1]) : exchange.amount})
    calculationSetup = {'inv' : activities_list, 'ia' : method}
    bd.calculation_setups['breakdown'] = calculationSetup
    my_lca = bc.MultiLCA('breakdown')
    results = pd.DataFrame(my_lca.results.transpose(), columns = [str(list(i.keys())[0]).split('\'')[1] for i in activities_list], index = pd.MultiIndex.from_tuples(method))
    results = results.sort_index().drop(index = [i for i in results.index if i[0] == 'rest'])
    direct_emissions = pd.DataFrame([0 if abs(results.iloc[r,0] - results.iloc[r,1:].sum()) / abs(results.iloc[r,0]) < 1e-5 else results.iloc[r, 0] - results.iloc[r, 1:].sum() for r in range(len(results.index))],
                                        columns = ['direct emissions'],
                                        index = results.index)
    results = pd.concat([results, direct_emissions], axis = 1)
    return results

In [39]:
df = pd.DataFrame()

In [40]:
activities = [activity for activity in db if 'acetylene' in activity['reference product']]
activities

['vinyl chloride monomer production, fossil acetylene' (kilogram, GLO, None),
 'acetylene production, natural gas, hot plasma' (kilogram, GLO, None),
 'acetylene production, natural gas, thermal pyrolysis' (kilogram, GLO, None),
 'acetylene production, natural gas, cold plasma' (kilogram, GLO, None),
 'acetylene production, biomethane, thermal pyrolysis' (kilogram, GLO, None),
 'acetylene production, biomethane, thermal pyrolysis, wind electricity' (kilogram, GLO, None),
 'acetylene production, biomethane, partial oxidation, wind electricity' (kilogram, GLO, None),
 'acetylene production, biomethane, cold plasma, wind electricity' (kilogram, GLO, None),
 'acetylene production, biomethane, hot plasma, wind electricity' (kilogram, GLO, None),
 'acetylene production, biomethane, partial oxidation' (kilogram, GLO, None),
 'acetylene production, biochar, calcium carbide, wind electricity' (kilogram, GLO, None),
 'acetylene production, biomethane, hot plasma' (kilogram, GLO, None),
 'acetyle

In [41]:
for activity in activities:
    df = pd.concat([df, breakdown_calculations(db, activity)], ignore_index = True)

TypeError: MultiLCA.__init__() got an unexpected keyword argument 'use_distributions'

In [10]:
df.fillna(0, inplace = True)
df

,"acetylene production, biomethane, cold plasma","biomethane production, from biogas upgrading, using amine scrubbing","market group for electricity, high voltage",direct emissions,"acetylene production, natural gas, partial oxidation","market group for natural gas, high pressure",market for N-methyl-2-pyrrolidone,"market for steam, in chemical industry","market for water, deionised","acetylene production, biomethane, thermal pyrolysis, wind electricity",...,"market for sodium hypochlorite, without water, in 15% solution state","market for sodium hydroxide, without water, in 50% solution state","market for nitrogen, liquid","acetylene production, natural gas, cold plasma","acetylene production, coal, calcium carbide","calcium carbide production, coal","acetylene production, natural gas, hot plasma","acetylene production, biomethane, thermal pyrolysis","acetylene production, biochar, calcium carbide, wind electricity","calcium carbide production, biochar, wind electricity"
0,5.004879,-3.585845,8.590724,0.00000,0.00000,0.000000,0.000000,0.000000,0.000000,0.00000,...,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000
1,0.000000,0.000000,2.239700,0.80000,7.81265,3.135920,0.066311,1.569122,0.001597,0.00000,...,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000
2,0.000000,-3.266026,0.000000,0.00000,0.00000,0.000000,0.000000,0.000000,0.000000,-2.12541,...,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000
3,0.000000,0.000000,0.000000,0.00000,0.00000,0.947812,0.000000,0.000000,0.000000,0.00000,...,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000
4,0.000000,-10.805943,0.000000,0.80000,0.00000,0.000000,0.066311,1.569122,0.001597,0.00000,...,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000
5,0.000000,-3.585845,0.000000,0.00000,0.00000,0.000000,0.000000,0.000000,0.000000,0.00000,...,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000
6,0.000000,-3.765137,8.217214,0.00000,0.00000,0.000000,0.061890,1.394776,0.003090,0.00000,...,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000
7,0.000000,-3.765137,0.000000,0.00000,0.00000,0.000000,0.061890,1.394776,0.003090,0.00000,...,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000
8,0.000000,-10.805943,2.239700,0.80000,0.00000,0.000000,0.066311,1.569122,0.001597,0.00000,...,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000
9,0.000000,0.000000,0.343018,0.00144,0.00000,0.000000,0.000000,0.000000,0.003193,0.00000,...,0.117485,0.002317,0.0,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000


In [11]:
results_file_path = os.path.join('..', 'results', 'acetylene_climate_change_impact_breakdown_2020.xlsx')
with pd.ExcelWriter(results_file_path, engine='xlsxwriter') as writer:
    df.to_excel(writer, sheet_name = 'overall')

In [12]:
df = pd.DataFrame()

In [13]:
activities = [activity for activity in db if 'calcium carbide production' in activity['name']]
activities

['calcium carbide production, biochar, wind electricity' (kilogram, GLO, None),
 'calcium carbide production, biochar' (kilogram, GLO, None),
 'calcium carbide production, coal' (kilogram, GLO, None)]

In [14]:
for activity in activities:
    df = pd.concat([df, breakdown_calculations(db, activity)], ignore_index = True)

In [15]:
df.fillna(0, inplace = True)
df

,"calcium carbide production, biochar, wind electricity","market for biochar, wind electricity","market for limestone, unprocessed","electricity production, wind, >3MW turbine, onshore","market for water, deionised","market for compressed air, 800 kPa gauge","market for nitrogen, liquid",direct emissions,"calcium carbide production, biochar",market for biochar,"market group for electricity, high voltage","calcium carbide production, coal",market for coke
0,-1.588841,-2.888231,0.003737,0.122134,0.001047,0.011182,0.0,1.16129,0.000000,0.000000,0.000000,0.000000,0.000000
1,0.000000,0.000000,0.003737,0.000000,0.001047,0.011182,0.0,1.16129,0.944751,-2.488832,2.256327,0.000000,0.000000
2,0.000000,0.000000,0.003737,0.000000,0.001047,0.011182,0.0,1.16129,0.000000,0.000000,2.256327,4.026578,0.592996


In [16]:
results_file_path = os.path.join('..', 'results', 'calcium_carbide_climate_change_impact_breakdown_2020.xlsx')
with pd.ExcelWriter(results_file_path, engine='xlsxwriter') as writer:
    df.to_excel(writer, sheet_name = 'overall')

In [17]:
db = bd.Database('acetylene-production-RCP19-2050')

In [18]:
df = pd.DataFrame()

In [19]:
activities = [activity for activity in db if 'acetylene' in activity['reference product']]
activities

['acetylene production, biomethane, thermal pyrolysis' (kilogram, GLO, None),
 'acetylene production, biomethane, hot plasma, wind electricity' (kilogram, GLO, None),
 'acetylene production, biochar, calcium carbide' (kilogram, GLO, None),
 'acetylene production, biomethane, partial oxidation, wind electricity' (kilogram, GLO, None),
 'acetylene production, natural gas, partial oxidation' (kilogram, GLO, None),
 'acetylene production, natural gas, thermal pyrolysis' (kilogram, GLO, None),
 'acetylene production, natural gas, hot plasma' (kilogram, GLO, None),
 'acetylene production, biomethane, partial oxidation' (kilogram, GLO, None),
 'acetylene production, biomethane, thermal pyrolysis, wind electricity' (kilogram, GLO, None),
 'acetylene production, coal, calcium carbide' (kilogram, GLO, None),
 'acetylene production, biochar, calcium carbide, wind electricity' (kilogram, GLO, None),
 'acetylene production, biomethane, cold plasma' (kilogram, GLO, None),
 'acetylene production, bio

In [20]:
for activity in activities:
    df = pd.concat([df, breakdown_calculations(db, activity)], ignore_index = True)

In [21]:
df.fillna(0, inplace = True)
df

,"acetylene production, biomethane, thermal pyrolysis","heat production, natural gas, at boiler condensing modulating >100kW","biomethane production, from biogas upgrading, using amine scrubbing",direct emissions,"acetylene production, biomethane, hot plasma, wind electricity",market for N-methyl-2-pyrrolidone,"electricity production, wind, >3MW turbine, onshore","market for steam, in chemical industry","market for water, deionised","acetylene production, biochar, calcium carbide",...,"acetylene production, biomethane, partial oxidation","acetylene production, biomethane, thermal pyrolysis, wind electricity","acetylene production, coal, calcium carbide","calcium carbide production, coal","acetylene production, biochar, calcium carbide, wind electricity","calcium carbide production, biochar, wind electricity","acetylene production, biomethane, cold plasma","acetylene production, biomethane, hot plasma","acetylene production, biomethane, cold plasma, wind electricity","acetylene production, natural gas, cold plasma"
0,-2.366096,1.10054,-3.466635,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1,0.000000,0.00000,-3.996403,0.00000,-2.472135,0.036445,0.206514,1.279971,0.001338,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,0.000000,0.00000,0.000000,0.00144,0.000000,0.000000,0.000000,0.000000,0.001383,-6.199842,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
3,0.000000,0.00000,-11.469676,0.80000,0.000000,0.039048,0.056288,1.439968,0.000692,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
4,0.000000,0.00000,0.000000,0.80000,0.000000,0.039048,0.000000,1.439968,0.000692,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
5,0.000000,1.10054,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
6,0.000000,0.00000,0.000000,0.00000,0.000000,0.036445,0.000000,1.279971,0.001338,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
7,0.000000,0.00000,-11.469676,0.80000,0.000000,0.039048,0.000000,1.439968,0.000692,0.000000,...,-9.337565,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
8,0.000000,1.10054,-3.466635,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,-2.366096,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
9,0.000000,0.00000,0.000000,0.00144,0.000000,0.000000,0.000000,0.000000,0.001383,0.000000,...,0.000000,0.000000,4.776244,4.779732,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


In [22]:
results_file_path = os.path.join('..', 'results', 'acetylene_climate_change_impact_breakdown_2050.xlsx')
with pd.ExcelWriter(results_file_path, engine='xlsxwriter') as writer:
    df.to_excel(writer, sheet_name = 'overall')

In [23]:
df = pd.DataFrame()

In [24]:
activities = [activity for activity in db if 'calcium carbide production' in activity['name']]
activities

['calcium carbide production, coal' (kilogram, GLO, None),
 'calcium carbide production, biochar' (kilogram, GLO, None),
 'calcium carbide production, biochar, wind electricity' (kilogram, GLO, None)]

In [25]:
for activity in activities:
    df = pd.concat([df, breakdown_calculations(db, activity)], ignore_index = True)

In [26]:
df.fillna(0, inplace = True)
df

,"calcium carbide production, coal","market for limestone, unprocessed",market for coke,"market group for electricity, high voltage","market for water, deionised","market for compressed air, 800 kPa gauge","market for nitrogen, liquid",direct emissions,"calcium carbide production, biochar",market for biochar,"calcium carbide production, biochar, wind electricity","market for biochar, wind electricity","electricity production, wind, >3MW turbine, onshore"
0,1.541849,0.002375,0.52548,-0.148692,0.000453,0.000941,0.0,1.16129,0.000000,0.000000,0.000000,0.000000,0.000000
1,0.000000,0.002375,0.00000,-0.148692,0.000453,0.000941,0.0,1.16129,-1.998824,-3.015193,0.000000,0.000000,0.000000
2,0.000000,0.002375,0.00000,0.000000,0.000453,0.000941,0.0,1.16129,0.000000,0.000000,-1.754988,-2.976754,0.056706


In [27]:
results_file_path = os.path.join('..', 'results', 'calcium_carbide_climate_change_impact_breakdown_2050.xlsx')
with pd.ExcelWriter(results_file_path, engine='xlsxwriter') as writer:
    df.to_excel(writer, sheet_name = 'overall')